In [0]:
%python
import sys
sys.path.append("/Workspace/Users/alliesdl@gmail.com/MLops_bal_pipeline")   # your Git folder
from pyspark.sql import functions as F, Window
from src.cleaning import *
from src.quality import assert_unique, unparsed_dates, null_report

POS_NEG = {"pos": "Positive", "positive": "Positive",
           "neg": "Negative", "negative": "Negative"}
IGRA = {**POS_NEG, "indeterminate": "Indeterminate", "borderline": "Borderline"}
RIF = {"sensitive": "Sensitive", "resistant": "Resistant", "indeterminate": "Indeterminate",
       "nd": "Not tested", "not applicable": "Not tested"}
SMOKING = {"yes": "Current", "no": "Never", "never": "Never",
           "ex": "Former", "ex-smoker": "Former", "former": "Former"}  # assumes "no" = never smoked: confirm

m = spark.table("clinical_portfolio.bronze.metadata")

# 1. Look before you map
report_unmapped(m, "smear_grade", SMEAR).show()

# 2. Clean
m = clean_id(m)
m = parse_date(m, "date_seen")
m = parse_date(m, "birthdate", lo="1920-01-01", hi="2015-12-31")
for col, mapping in [("sex", SEX), ("smear_grade", SMEAR), ("cxr_finding", CXR),
                     ("smoking_status", SMOKING), ("igra_result", IGRA),
                     ("hiv_status", POS_NEG), ("drug_use", YES_NO),
                     ("vaping", YES_NO), ("art", YES_NO), ("genexpert_rif", RIF)]:
    m = map_values(m, col, mapping)
for col in ["height_1_cm", "height_2_cm", "weight_1_kg", "weight_2_kg",
            "cd4_count", "viral_load_cp_ml", "igra_value_iu_ml"]:
    m = parse_number(m, col)

# 3. Impossible values (zero, negative, 999 codes, 1863 cm) become nulls
for col, lo, hi in [("height_1_cm", 100, 220), ("height_2_cm", 100, 220),
                    ("weight_1_kg", 10, 250), ("weight_2_kg", 10, 250)]:
    m = null_outside(m, col, lo, hi)

# 4. Derive, then remove direct identifiers
m = (m.withColumn("age_years", F.floor(F.months_between("date_seen", "birthdate") / 12))
       .withColumn("height_cm", F.expr("(coalesce(height_1_cm, height_2_cm) + coalesce(height_2_cm, height_1_cm)) / 2"))
       .withColumn("weight_kg", F.expr("(coalesce(weight_1_kg, weight_2_kg) + coalesce(weight_2_kg, weight_1_kg)) / 2"))
       .withColumn("bmi", F.col("weight_kg") / F.pow(F.col("height_cm") / 100, 2))
       .drop("surname", "firstname", "birthdate", "birthdate_raw",
             "height_1_cm", "height_2_cm", "weight_1_kg", "weight_2_kg"))

# 5. One row per participant: keep the latest visit date
w = Window.partitionBy("pt_id").orderBy(F.col("date_seen").desc_nulls_last())
m = m.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

# 6. Quality gate: stop here if something is wrong
assert_unique(m, "pt_id")
assert unparsed_dates(m, "date_seen") == 0
print(null_report(m))

(m.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("clinical_portfolio.silver.metadata"))

In [0]:
%python
b = (spark.table("clinical_portfolio.bronze.bal")
          .withColumnRenamed("_source_sheet", "visit"))
b = clean_id(b)
b = b.filter("pt_id IS NOT NULL")                 # the blank spreadsheet rows
b = parse_date(b, "bronc_date")

NUMERIC = ["vol_pbs_in_ml", "vol_pbs_out_ml", "live_cells", "dead_cells", "total_cells",
           "time_to_process_min", "macrophages_pct", "lymphocytes_pct", "neutrophils_pct",
           "eosinophils_pct", "basophils_pct", "mastcells_pct"]
for c in NUMERIC:
    b = parse_number(b, c)

b = null_outside(b, "live_cells", 0, 1e8)         # negative cell counts
b = null_outside(b, "time_to_process_min", 1, 600)  # zero-minute processing times

# Spark's strict mode stops the job on divide-by-zero, so use try_divide (gives null instead)
b = (b.withColumn("viability_pct",
                  F.expr("try_divide(live_cells, live_cells + dead_cells) * 100"))
      .withColumn("recovery_ratio", F.expr("try_divide(vol_pbs_out_ml, vol_pbs_in_ml)")))

assert_unique(b, ["pt_id", "visit"])
print(b.count(), b.groupBy("visit").count().collect())
(b.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("clinical_portfolio.silver.bal"))

In [0]:
%python
t = spark.table("clinical_portfolio.bronze.treatment")
t = clean_id(t)
t = parse_date(t, "treatment_start_date")
t = (map_values(t, "treatment_outcome", OUTCOME_GROUP, keep_unmapped=False)
       .withColumnRenamed("treatment_outcome", "outcome_group"))
w = Window.partitionBy("pt_id").orderBy(F.col("treatment_start_date").desc_nulls_last())
t = t.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
assert_unique(t, "pt_id")
(t.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
  .saveAsTable("clinical_portfolio.silver.treatment"))